### 樣板文字（模板）偵測

針對 `1_llm_ready...` 產出的 snippet-level CSV，標記出免責聲明、評等機構制式用語等樣板文字，用兩種方法合併判斷，**不刪除原始資料，只新增欄位**：

1. **頻率法**：同一段 `llm_input_text` 出現在 ≥ `FREQ_THRESHOLD` 篇不同文章 → 判定為樣板
2. **關鍵字法**：比對 `template_phrases.csv` 字典，算「比對到的字數 / snippet 總字數」的重疊比例，≥ `KEYWORD_OVERLAP_THRESHOLD` → 判定為樣板

> `template_phrases.csv` 目前是憑常識草擬的起始版本，**還沒被真實資料驗證過**，請務必看過 cell 4 印出的邊界樣本，確認門檻/字典準不準，必要時回頭調整。

#### 1. 路徑與門檻設定

In [1]:
import os

year = 2024
snippet_csv = (
    r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments'
    rf'\output\llm_ready_整年\{year}\llm_ready_data_{year}_context50_snippetlevel.csv'
)

phrase_dict_csv = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\template_phrases.csv'

output_dir = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output'
os.makedirs(output_dir, exist_ok=True)
output_csv = os.path.join(output_dir, f'{year}_snippet_flagged.csv')

# 頻率法：同一段文字出現在幾篇不同文章以上，判定為樣板
FREQ_THRESHOLD = 5

# 關鍵字法：比對到的字數 / snippet 總字數 的重疊比例門檻（起始值，需依 cell 4 的邊界樣本調整）
KEYWORD_OVERLAP_THRESHOLD = 0.2

print('snippet_csv:', snippet_csv)
print('phrase_dict_csv:', phrase_dict_csv)
print('output_csv:', output_csv)

snippet_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv
phrase_dict_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\template_phrases.csv
output_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_snippet_flagged.csv


#### 2. 載入模板字典 + 定義清洗/比對函式

In [ ]:
import re
import pandas as pd

def clean_text(text):
    """跟 1_llm_ready 的 process_single_txt 用同一套清洗規則，確保跟已清洗過的 llm_input_text 對得上"""
    return re.sub(r'[^a-zA-Z\s]', ' ', str(text)).lower().strip()


phrase_df = pd.read_csv(phrase_dict_csv)
phrase_df['phrase_clean'] = phrase_df['phrase'].apply(clean_text)
phrase_df['phrase_word_count'] = phrase_df['phrase_clean'].str.split().apply(len)
phrase_list = list(zip(phrase_df['phrase_clean'], phrase_df['phrase_word_count'], phrase_df['template_type']))
print(f"已載入 {len(phrase_list)} 個模板文字段")
print(phrase_df['template_type'].value_counts())


def compute_keyword_overlap(text):
    """回傳 (重疊比例, 命中字數最多的 template_type)。text 需已清洗過的小寫字串。"""
    total_words = len(text.split())
    if total_words == 0:
        return 0.0, None
    type_word_counts = {}
    # 比對每個模板文字段，計算命中字數
    for phrase, word_count, ttype in phrase_list:
        if phrase and  f" {phrase} " in f" {text} ":
            type_word_counts[ttype] = type_word_counts.get(ttype,
                                                           0) + word_count
    if not type_word_counts:
        return 0.0, None
    matched_words = sum(type_word_counts.values())
    best_type = max(type_word_counts, key=type_word_counts.get)
    # 避免重疊比例超過 1.0
    return min(matched_words / total_words, 1.0), best_type

已載入 59 個模板文字段
template_type
disclaimer    33
rating        10
copyright      8
website        8
Name: count, dtype: int64


#### 3. 對「唯一文字」算一次頻率法 + 關鍵字法（避免對 2 百萬列重複計算）

In [3]:
import time

print(f"讀取 snippet 檔案: {snippet_csv}")
df_snippet = pd.read_csv(snippet_csv)
print(f"總列數: {len(df_snippet):,}")

t0 = time.time()
text_stats = df_snippet.groupby('llm_input_text').agg(
    dup_article_count=('file_name', 'nunique'),
).reset_index()
print(f"唯一文字數: {len(text_stats):,}")

# 頻率法
text_stats['freq_flag'] = text_stats['dup_article_count'] >= FREQ_THRESHOLD

# 關鍵字法（只對唯一文字算一次；llm_input_text 理論上已經清洗過，這裡用 clean_text 再過一次確保一致）
overlap_results = text_stats['llm_input_text'].apply(clean_text).apply(compute_keyword_overlap)
text_stats['keyword_overlap_pct'] = overlap_results.apply(lambda x: x[0])
text_stats['keyword_type'] = overlap_results.apply(lambda x: x[1])
text_stats['keyword_flag'] = text_stats['keyword_overlap_pct'] >= KEYWORD_OVERLAP_THRESHOLD

print(f"關鍵字比對耗時: {time.time() - t0:.1f} 秒")


def combine_flags(row):
    freq, kw = row['freq_flag'], row['keyword_flag']
    if freq and kw:
        return True, 'both', row['keyword_type'] if row['keyword_type'] else 'frequency'
    elif freq:
        return True, 'frequency', 'frequency'
    elif kw:
        return True, 'keyword', row['keyword_type']
    else:
        return False, None, None


combined = text_stats.apply(combine_flags, axis=1, result_type='expand')
combined.columns = ['is_template', 'flag_reason', 'template_type']
text_stats = pd.concat([text_stats, combined], axis=1)

# dup_article_count_out / keyword_overlap_pct 依 flag_reason 決定要不要留（另外開欄位）
text_stats['dup_article_count_out'] = text_stats.apply(
    lambda r: r['dup_article_count'] if r['flag_reason'] in ('frequency', 'both') else pd.NA, axis=1)
text_stats['keyword_overlap_pct_out'] = text_stats.apply(
    lambda r: round(r['keyword_overlap_pct'], 4) if r['flag_reason'] in ('keyword', 'both') else pd.NA, axis=1)

print(f"\n模板文字比例(依唯一文字計): {text_stats['is_template'].mean():.2%}")
print(text_stats['flag_reason'].value_counts(dropna=False))

讀取 snippet 檔案: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv
總列數: 2,207,756
唯一文字數: 939,278
關鍵字比對耗時: 30.1 秒

模板文字比例(依唯一文字計): 8.29%
flag_reason
None         861373
frequency     76579
keyword        1124
both            202
Name: count, dtype: int64


#### 4. 映射回原始 snippet 資料 + 診斷輸出（門檻邊界樣本、matched_bigram 比較）

In [4]:
lookup = text_stats[['llm_input_text', 'is_template', 'template_type', 'flag_reason',
                      'dup_article_count_out', 'keyword_overlap_pct_out']].rename(columns={
    'dup_article_count_out': 'dup_article_count',
    'keyword_overlap_pct_out': 'keyword_overlap_pct',
})

df_snippet = df_snippet.merge(lookup, on='llm_input_text', how='left')

print("=== 整體統計（依實際列數計） ===")
print(f"is_template 比例: {df_snippet['is_template'].mean():.2%}")
print(df_snippet['flag_reason'].value_counts(dropna=False))

print("\n=== 頻率法門檻邊界樣本（dup_article_count 在 FREQ_THRESHOLD±1 之間） ===")
boundary_freq = text_stats[text_stats['dup_article_count'].between(FREQ_THRESHOLD - 1, FREQ_THRESHOLD + 1)]
for _, row in boundary_freq.head(10).iterrows():
    print(f"[{row['dup_article_count']}篇] {row['llm_input_text'][:150]}")

print("\n=== 關鍵字法門檻邊界樣本（overlap 25%~35%） ===")
boundary_kw = text_stats[text_stats['keyword_overlap_pct'].between(0.25, 0.35)]
for _, row in boundary_kw.head(10).iterrows():
    print(f"[{row['keyword_overlap_pct']:.1%}, type={row['keyword_type']}] {row['llm_input_text'][:150]}")

print("\n=== matched_bigram 分布比較：樣板 vs 非樣板（檢查有沒有誤殺真正有意義的詞） ===")
print("樣板 snippet 裡最常見的 matched_bigram:")
print(df_snippet[df_snippet['is_template'] == True]['matched_bigram'].value_counts().head(15))
print("\n非樣板 snippet 裡最常見的 matched_bigram:")
print(df_snippet[df_snippet['is_template'] == False]['matched_bigram'].value_counts().head(15))

=== 整體統計（依實際列數計） ===
is_template 比例: 44.70%
flag_reason
None         1220967
frequency     982139
both            2950
keyword         1700
Name: count, dtype: int64

=== 頻率法門檻邊界樣本（dup_article_count 在 FREQ_THRESHOLD±1 之間） ===
[4篇] a adjusted ebitda margin as we look ahead to we re focused on executing against our growth initiatives for the long term yelp has not reconciled its a
[4篇] a alloimmunized an immune response to foreign antigens upon exposure to genetically different cells or tissues about the unity trial unity nct is a gl
[4篇] a and generating sustainable profitable growth and delivering value for shareholders over the long term because these forward looking statements invol
[4篇] a and share of revenues respectively industries with extensive supply and distribution networks for goods are the dominant investors in sccts but grow
[4篇] a at the university of pennsylvania mr bienaim will join a board of directors headed by experienced biopharmaceutical industry figures and leadin

#### 5. 輸出（新檔案，不覆蓋原始 snippet CSV）

In [5]:
# df_snippet.to_csv(output_csv, index=False, encoding='utf-8-sig')
# print(f"已輸出: {output_csv}")
# print(f"欄位: {df_snippet.columns.tolist()}")